In [1]:
import numpy as np
import json
from pathlib import Path
from datetime import datetime
import rasterio

# ============================================================
# CONFIGURATION
# ============================================================
ALIGNED_DATA_DIR = "/p/scratch/training2600/TeamGudnason/data/aligned_data"
OUTPUT_DIR       = "/p/scratch/training2600/TeamGudnason/training_data/final_block_64"

T33TYN_TILES = [
    "S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829",
    "S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859",
    "S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012",
    "S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408",
]

PATCH_SIZE = 64    # 64x64 pixels = 640m x 640m
BLOCK_SIZE = 640   # ~2x patch size til að forðast overlap við block jaðra
TRAIN_FRAC = 0.50
VAL_FRAC   = 0.25
TEST_FRAC  = 0.25

N_TRAIN = 10000
N_VAL   = 2000
N_TEST  = 2000

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("=" * 70)
print(f"BLOCK SPLIT WITH {PATCH_SIZE}x{PATCH_SIZE} PATCHES")
print("=" * 70)

# ============================================================
# Extract patches with coordinates
# ============================================================
def extract_patches_with_coords(s2_path, corine_path, patch_size=64,
                                 low_pct=2, high_pct=98):
    with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as corine_src:
        height = s2_src.height
        width  = s2_src.width
        stride = patch_size  # engin overlap

        s2_data     = s2_src.read()
        corine_data = corine_src.read(1)

        lo = np.percentile(s2_data, low_pct)
        hi = np.percentile(s2_data, high_pct)
        s2_norm = np.clip((s2_data.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)
        norm_params = {'method': 'percentile', 'low_pct': low_pct, 'high_pct': high_pct,
                       'low_val': float(lo), 'high_val': float(hi)}

        coord_to_patch = {}
        coord_to_label = {}

        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                center_y = y + patch_size // 2
                center_x = x + patch_size // 2

                # Use majority label in patch center region (8x8)
                margin = patch_size // 8
                center_region = corine_data[
                    center_y - margin:center_y + margin,
                    center_x - margin:center_x + margin
                ]
                valid_labels = center_region[(center_region >= 1) & (center_region <= 44)]
                if len(valid_labels) == 0:
                    continue

                # Majority vote for label
                label = np.bincount(valid_labels.astype(int)).argmax()
                if label < 1 or label > 44:
                    continue

                # Skip if too many zero pixels
                orig_patch = s2_data[:, y:y+patch_size, x:x+patch_size]
                zero_frac = np.mean(orig_patch == 0)
                if zero_frac > 0.1:
                    continue

                patch = s2_norm[:, y:y+patch_size, x:x+patch_size]
                patch = np.transpose(patch, (1, 2, 0))  # (H, W, C)

                coord_to_patch[(center_y, center_x)] = patch
                coord_to_label[(center_y, center_x)] = label

    return coord_to_patch, coord_to_label, height, width, norm_params

# ============================================================
# Load all tiles
# ============================================================
print("\n[1/4] Building coordinate maps...")
data_path = Path(ALIGNED_DATA_DIR)
tile_dicts, label_dicts, norm_params_all = [], [], []
img_height, img_width = None, None

for tile_name in T33TYN_TILES:
    s2_path     = data_path / f"{tile_name}_stacked.tif"
    corine_path = data_path / f"corine_aligned_{tile_name}.tif"
    print(f"  {tile_name[7:22]}...")
    c2p, c2l, h, w, np_ = extract_patches_with_coords(s2_path, corine_path, patch_size=PATCH_SIZE)
    print(f"    -> {len(c2p):,} patches")
    tile_dicts.append(c2p)
    label_dicts.append(c2l)
    norm_params_all.append(np_)
    if img_height is None:
        img_height = h
        img_width  = w

# ============================================================
# Merge into time series
# ============================================================
print("\n[2/4] Merging into time series...")
common_coords = set(tile_dicts[0].keys())
for td in tile_dicts[1:]:
    common_coords &= set(td.keys())
print(f"  Common locations: {len(common_coords):,}")

common_coords = sorted(common_coords)
ts_patches, ts_labels, ts_rows, ts_cols = [], [], [], []

for (row, col) in common_coords:
    band_stack = np.concatenate(
        [tile_dicts[t][(row, col)] for t in range(len(T33TYN_TILES))],
        axis=-1
    )  # (64, 64, 16)
    ts_patches.append(band_stack)
    ts_labels.append(label_dicts[0][(row, col)])
    ts_rows.append(row)
    ts_cols.append(col)

ts_patches = np.array(ts_patches, dtype=np.float32)
ts_labels  = np.array(ts_labels,  dtype=np.uint8)
ts_rows    = np.array(ts_rows,    dtype=np.int32)
ts_cols    = np.array(ts_cols,    dtype=np.int32)

print(f"  Dataset: {ts_patches.shape}")

# ============================================================
# Block split
# ============================================================
print(f"\n[3/4] Block split (block size: {BLOCK_SIZE}px)...")

block_row = ts_rows // BLOCK_SIZE
block_col = ts_cols // BLOCK_SIZE
n_blocks_y = img_height // BLOCK_SIZE
n_blocks_x = img_width  // BLOCK_SIZE
total_blocks = n_blocks_y * n_blocks_x
print(f"  Grid: {n_blocks_x} x {n_blocks_y} = {total_blocks} blocks")

rng = np.random.default_rng(42)
block_ids = np.arange(total_blocks)
rng.shuffle(block_ids)

n_train_blocks = int(total_blocks * TRAIN_FRAC)
n_val_blocks   = int(total_blocks * VAL_FRAC)

train_blocks = set(block_ids[:n_train_blocks])
val_blocks   = set(block_ids[n_train_blocks:n_train_blocks + n_val_blocks])
test_blocks  = set(block_ids[n_train_blocks + n_val_blocks:])

patch_block_id = block_row * n_blocks_x + block_col
train_mask = np.array([int(b) in train_blocks for b in patch_block_id])
val_mask   = np.array([int(b) in val_blocks   for b in patch_block_id])
test_mask  = np.array([int(b) in test_blocks  for b in patch_block_id])

print(f"  Available: train={train_mask.sum():,}, val={val_mask.sum():,}, test={test_mask.sum():,}")

def sample_split(patches, labels, mask, n, rng):
    idx = np.where(mask)[0]
    rng.shuffle(idx)
    idx = idx[:n]
    return patches[idx], labels[idx]

train_patches, train_labels = sample_split(ts_patches, ts_labels, train_mask, N_TRAIN, rng)
val_patches,   val_labels   = sample_split(ts_patches, ts_labels, val_mask,   N_VAL,   rng)
test_patches,  test_labels  = sample_split(ts_patches, ts_labels, test_mask,  N_TEST,  rng)

print(f"\n  Final: train={train_patches.shape}, val={val_patches.shape}, test={test_patches.shape}")

# Class distribution
def label_dist(labels):
    unique, counts = np.unique(labels, return_counts=True)
    return {int(k): int(v) for k, v in zip(unique, counts)}

print("\n  Class distribution:")
tr_dist = label_dist(train_labels)
te_dist = label_dist(test_labels)
all_cls = sorted(set(tr_dist) | set(te_dist))
print(f"  {'CORINE':<10} {'Train%':>8} {'Test%':>8}")
for c in all_cls:
    tr_pct = 100 * tr_dist.get(c, 0) / len(train_labels)
    te_pct = 100 * te_dist.get(c, 0) / len(test_labels)
    if tr_pct > 0.5 or te_pct > 0.5:
        print(f"  {c:<10} {tr_pct:>8.1f}% {te_pct:>8.1f}%")

# ============================================================
# Save
# ============================================================
print("\n[4/4] Saving...")
out = Path(OUTPUT_DIR)

np.savez_compressed(out / "patches_train.npz", patches=train_patches)
np.savez_compressed(out / "patches_val.npz",   patches=val_patches)
np.savez_compressed(out / "patches_test.npz",  patches=test_patches)
np.save(out / "labels_train.npy", train_labels)
np.save(out / "labels_val.npy",   val_labels)
np.save(out / "labels_test.npy",  test_labels)

metadata = {
    "extraction_timestamp": datetime.now().isoformat(),
    "patch_size": PATCH_SIZE,
    "block_size": BLOCK_SIZE,
    "n_bands_total": 16,
    "splits": {
        "train": {"n_patches": len(train_labels), "label_distribution": label_dist(train_labels)},
        "val":   {"n_patches": len(val_labels),   "label_distribution": label_dist(val_labels)},
        "test":  {"n_patches": len(test_labels),  "label_distribution": label_dist(test_labels)},
    }
}
with open(out / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("  ✅ Vistað")
print(f"\nOutput: {OUTPUT_DIR}")
for fname in ["patches_train.npz", "patches_val.npz", "patches_test.npz"]:
    fpath = out / fname
    if fpath.exists():
        print(f"  {fname}: {fpath.stat().st_size/(1024**2):.1f} MB")

BLOCK SPLIT WITH 64x64 PATCHES

[1/4] Building coordinate maps...
  L2A_20180408T09...
    -> 29,241 patches
  L2A_20180523T09...
    -> 29,241 patches
  L2A_20180816T09...
    -> 29,241 patches
  L2A_20181020T09...
    -> 29,241 patches

[2/4] Merging into time series...
  Common locations: 29,241
  Dataset: (29241, 64, 64, 16)

[3/4] Block split (block size: 640px)...
  Grid: 17 x 17 = 289 blocks
  Available: train=14,450, val=7,260, test=7,350

  Final: train=(10000, 64, 64, 16), val=(2000, 64, 64, 16), test=(2000, 64, 64, 16)

  Class distribution:
  CORINE       Train%    Test%
  2               7.8%      6.6%
  3               2.1%      1.2%
  11              1.0%      1.6%
  12             50.7%     45.4%
  15              1.1%      1.1%
  16              0.7%      0.6%
  18              5.2%      5.8%
  20              2.4%      3.1%
  21              1.8%      1.4%
  23             15.0%     19.6%
  24              0.5%      0.7%
  25              0.9%      1.4%
  26          